## 2.1 Load and Combine the Dataset

The following cells load the seven CICIDS2017 CSV files (Tuesday through
Friday, excluding Monday's benign-only file), strip whitespace from column
names, and combine them into a single DataFrame with a source_day column
for traceability.

In [1]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "data"

FILES = [
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

In [2]:
def load_csv(fname):
    path = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(path, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.strip()
    df["source_day"] = fname.split("_")[0].split(".")[0]
    return df
    
frames = []

for fname in FILES:
    df = load_csv(fname)
    frames.append(df)
    print(f"Loaded {fname}: {df.shape}")

Loaded Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 80)
Loaded Wednesday-workingHours.pcap_ISCX.csv: (692703, 80)
Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (170366, 80)
Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (288602, 80)
Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: (191033, 80)
Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (286467, 80)
Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (225745, 80)


In [3]:
# Combine datasets
data = pd.concat(frames, ignore_index=True)
print("Combined shape:", data.shape)

# Initial checks
print("Duplicate columns detected:", [c for c in data.columns if c.endswith(".1")])

Combined shape: (2300825, 80)
Duplicate columns detected: ['Fwd Header Length.1']


## 2.2 Exploratory Data Analysis

### 2.2.1 Class Distribution

In [4]:
# Class distribution and imbalance ratio highlighting the disparity between majority and minority traffic classes.

import gc
print("\n=== Class distribution ===")
print(data["Label"].value_counts())
 
imbalance_ratio = data["Label"].value_counts().max() / data["Label"].value_counts().min()
print("\nImbalance ratio (largest class / smallest class):", round(imbalance_ratio, 1))


=== Class distribution ===
Label
BENIGN                          1743179
DoS Hulk                         231073
PortScan                         158930
DDoS                             128027
DoS GoldenEye                     10293
FTP-Patator                        7938
SSH-Patator                        5897
DoS slowloris                      5796
DoS Slowhttptest                   5499
Bot                                1966
Web Attack ï¿½ Brute Force         1507
Web Attack ï¿½ XSS                  652
Infiltration                         36
Web Attack ï¿½ Sql Injection         21
Heartbleed                           11
Name: count, dtype: int64

Imbalance ratio (largest class / smallest class): 158470.8


### 2.2.2 Data Quality Issues

In [5]:
# Summary of missing and infinite values identifying columns with NaNs and features affected by non‑finite values prior to cleaning.

print("\n=== Missing values (columns with any NaN) ===")
na_counts = data.isna().sum()
print(na_counts[na_counts > 0])
 
print("\n=== Infinite values ===")
numeric_cols = data.select_dtypes(include=[np.number]).columns
inf_report = {}
for c in numeric_cols:
    n = np.isinf(data[c]).sum()
    if n > 0:
        inf_report[c] = int(n)
print(inf_report)


=== Missing values (columns with any NaN) ===
Flow Bytes/s    1294
dtype: int64

=== Infinite values ===
{'Flow Bytes/s': 1136, 'Flow Packets/s': 2430}


In [6]:
# Hash‑based duplicate row analysis reporting the absolute number and percentage of fully duplicated feature rows after excluding metadata and label columns.


print("\n=== Duplicate rows ===")
feature_cols = [c for c in data.columns if c not in ("source_day", "Label")]
row_hashes = pd.util.hash_pandas_object(data[feature_cols], index=False)
dup_mask = row_hashes.duplicated(keep="first")
print("Duplicate rows:", int(dup_mask.sum()))
print("As % of total:", round(100 * dup_mask.sum() / len(data), 2), "%")
del row_hashes, dup_mask
_ = gc.collect()


=== Duplicate rows ===
Duplicate rows: 281791
As % of total: 12.25 %


### 2.2.3 Feature Correlation and Leakage Risk

In [7]:
# Detection of zero‑variance features and duplicated columns identifying non‑informative predictors and confirming that Fwd Header Length.1 is identical to Fwd Header Length.


print("\n=== Zero-variance columns (candidates to drop) ===")
zero_var_cols = [c for c in numeric_cols if data[c].nunique() <= 1]
print(zero_var_cols)
 
print("\n=== Duplicate feature columns ===")
dup_feature_cols = [c for c in data.columns if c.endswith(".1")]
print(dup_feature_cols)
if "Fwd Header Length.1" in data.columns:
    identical = (data["Fwd Header Length"] == data["Fwd Header Length.1"]).all()
    print("Fwd Header Length == Fwd Header Length.1 ?", identical)


=== Zero-variance columns (candidates to drop) ===
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

=== Duplicate feature columns ===
['Fwd Header Length.1']
Fwd Header Length == Fwd Header Length.1 ? True


In [8]:
# Most frequent destination port for each traffoc class revealing strong one‑to‑one mappings between specific ports and attack types and highlighting the risk of label leakage if Destination Port is used as a predictor.
print("\n=== Most common Destination Port per class ===")
top_port_per_class = data.groupby("Label")["Destination Port"].agg(
    lambda x: x.value_counts().index[0]
)
print(top_port_per_class)
print("\nNote: several attack classes map almost entirely onto a single port")
print("(e.g. FTP-Patator -> 21, SSH-Patator -> 22, DoS/DDoS/PortScan/Web/Bot -> 80 or 8080).")
print("This is a real risk of data leakage: a model could learn to classify by port")


=== Most common Destination Port per class ===
Label
BENIGN                            53
Bot                             8080
DDoS                              80
DoS GoldenEye                     80
DoS Hulk                          80
DoS Slowhttptest                  80
DoS slowloris                     80
FTP-Patator                       21
Heartbleed                       444
Infiltration                     444
PortScan                          80
SSH-Patator                       22
Web Attack ï¿½ Brute Force        80
Web Attack ï¿½ Sql Injection      80
Web Attack ï¿½ XSS                80
Name: Destination Port, dtype: int64

Note: several attack classes map almost entirely onto a single port
(e.g. FTP-Patator -> 21, SSH-Patator -> 22, DoS/DDoS/PortScan/Web/Bot -> 80 or 8080).
This is a real risk of data leakage: a model could learn to classify by port


## 3.1 Cleaning

The following cells apply the cleaning steps justified in Section 2, in
order: correcting the Web Attack label encoding artefact, dropping the
three statistically unworkable classes, removing redundant columns,
handling infinite and missing values, and removing duplicate rows.

In [9]:
# Fix Web Attack label encoding artifact 
 
data["Label"] = data["Label"].str.strip()
 
data.loc[
    data["Label"].str.contains("Web Attack", na=False) & data["Label"].str.contains("Brute Force"),
    "Label"
] = "Web Attack - Brute Force"
 
data.loc[
    data["Label"].str.contains("Web Attack", na=False) & data["Label"].str.contains("XSS"),
    "Label"
] = "Web Attack - XSS"
 
data.loc[
    data["Label"].str.contains("Web Attack", na=False) & data["Label"].str.contains("Sql Injection"),
    "Label"
] = "Web Attack - Sql Injection"
 
print("Unique labels after fix:")
print(sorted(data["Label"].unique()))

Unique labels after fix:
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [10]:
# Drop statistically unworkable classes 
 
classes_to_drop = ["Heartbleed", "Infiltration", "Web Attack - Sql Injection"]
before = len(data)
data = data[~data["Label"].isin(classes_to_drop)].reset_index(drop=True)
print(f"Dropped {before - len(data)} rows belonging to: {classes_to_drop}")
print("Shape after dropping unworkable classes:", data.shape)

Dropped 68 rows belonging to: ['Heartbleed', 'Infiltration', 'Web Attack - Sql Injection']
Shape after dropping unworkable classes: (2300757, 80)


In [11]:
# Drop redundant columns 
 
cols_to_drop = [
    "Fwd Header Length.1",
    "Bwd PSH Flags", "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
]
cols_to_drop = [c for c in cols_to_drop if c in data.columns]
data = data.drop(columns=cols_to_drop)
print(f"Dropped columns: {cols_to_drop}")
print("Shape after dropping redundant columns:", data.shape)

Dropped columns: ['Fwd Header Length.1', 'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Shape after dropping redundant columns: (2300757, 71)


In [12]:
# Handle infinite and missing values
 
import numpy as np
numeric_cols = data.select_dtypes(include=[np.number]).columns
data[numeric_cols] = data[numeric_cols].replace([np.inf, -np.inf], np.nan)
 
before_na = len(data)
data = data.dropna(subset=numeric_cols).reset_index(drop=True)
print(f"Dropped {before_na - len(data)} rows containing inf/NaN values")
print("Shape after handling inf/NaN:", data.shape)

Dropped 2430 rows containing inf/NaN values
Shape after handling inf/NaN: (2298327, 71)


In [13]:
# Remove duplicate rows
 
feature_cols = [c for c in data.columns if c not in ("source_day", "Label")]
before_dup = len(data)
row_hashes = pd.util.hash_pandas_object(data[feature_cols], index=False)
data = data[~row_hashes.duplicated(keep="first")].reset_index(drop=True)
print(f"Dropped {before_dup - len(data)} duplicate rows")
print("Final shape after all cleaning steps:", data.shape)

Dropped 280617 duplicate rows
Final shape after all cleaning steps: (2017710, 71)


In [14]:
# Final class distribution check 
print("\nFinal class distribution after cleaning:")
print(data["Label"].value_counts())


Final class distribution after cleaning:
Label
BENIGN                      1592639
DoS Hulk                     172811
DDoS                         128011
PortScan                      90130
DoS GoldenEye                 10286
FTP-Patator                    5931
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Bot                            1948
Web Attack - Brute Force       1470
Web Attack - XSS                652
Name: count, dtype: int64


## 3.2 Feature and Label Encoding

In [15]:
# Separate features and label 
 
X = data.drop(columns=["Label", "source_day"])
y = data["Label"]
 
print("Feature matrix shape:", X.shape)
print("Number of feature columns:", X.shape[1])

Feature matrix shape: (2017710, 69)
Number of feature columns: 69


In [16]:
# Encode the label column 
 
from sklearn.preprocessing import LabelEncoder
 
le = LabelEncoder()
y_encoded = le.fit_transform(y)
 
print("Class encoding:")
for i, cls in enumerate(le.classes_):
    print(i, "->", cls)

Class encoding:
0 -> BENIGN
1 -> Bot
2 -> DDoS
3 -> DoS GoldenEye
4 -> DoS Hulk
5 -> DoS Slowhttptest
6 -> DoS slowloris
7 -> FTP-Patator
8 -> PortScan
9 -> SSH-Patator
10 -> Web Attack - Brute Force
11 -> Web Attack - XSS


## 3.3 Train, Validation, and Test Split

In [17]:
# Stratified train / validation / test split
 
from sklearn.model_selection import train_test_split
 
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, stratify=y_encoded, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)
 
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (1412397, 69)
Validation shape: (302656, 69)
Test shape: (302657, 69)


In [18]:
# Scale features

import numpy as np
from sklearn.preprocessing import StandardScaler

# convert to float32 first, as a standalone step, rather than casting inline
# inside the scaler call - this avoids sklearn internally upcasting back to
# float64 during the mean/variance calculation, which was doubling memory use
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train scaled mean (should be ~0):", round(X_train_scaled.mean(), 4))
print("Train scaled std (should be ~1):", round(X_train_scaled.std(), 4))

Train scaled mean (should be ~0): 0.0
Train scaled std (should be ~1): 1.0


In [19]:
# Confirm class balance survived the split
import pandas as pd
 
print("Class counts in training set:")
print(pd.Series(le.inverse_transform(y_train)).value_counts())

Class counts in training set:
BENIGN                      1114847
DoS Hulk                     120968
DDoS                          89608
PortScan                      63091
DoS GoldenEye                  7200
FTP-Patator                    4152
DoS slowloris                  3769
DoS Slowhttptest               3660
SSH-Patator                    2253
Bot                            1364
Web Attack - Brute Force       1029
Web Attack - XSS                456
Name: count, dtype: int64


## Save Processed Data

This notebook ends here. The processed arrays, scaler, and label encoder
are saved to disk so the modelling notebook can load them directly without
repeating the loading, cleaning, and preprocessing steps.

In [20]:
# Save everything to disk for the modelling notebook 
 
import os
import joblib
 
os.makedirs("processed_data", exist_ok=True)
 
np.save("processed_data/X_train.npy", X_train_scaled)
np.save("processed_data/X_val.npy", X_val_scaled)
np.save("processed_data/X_test.npy", X_test_scaled)
np.save("processed_data/y_train.npy", y_train)
np.save("processed_data/y_val.npy", y_val)
np.save("processed_data/y_test.npy", y_test)

In [21]:
# save the encoder and scaler, both are needed to interpret results

joblib.dump(le, "processed_data/label_encoder.joblib")
joblib.dump(scaler, "processed_data/scaler.joblib")
 
# save feature column names, needed later for feature importance plots
joblib.dump(list(X.columns), "processed_data/feature_names.joblib")
 
print("All processed data and objects saved to processed_data/")
print(os.listdir("processed_data"))

All processed data and objects saved to processed_data/
['feature_names.joblib', 'label_encoder.joblib', 'scaler.joblib', 'X_test.npy', 'X_train.npy', 'X_val.npy', 'y_test.npy', 'y_train.npy', 'y_val.npy']
